In [1]:
!pip install -U "mcp[cli]" google-genai httpx nest_asyncio


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import sys
import httpx
import nest_asyncio
nest_asyncio.apply()

from getpass import getpass
from google import genai
from google.genai import types

In [4]:
GEMINI_API_KEY = getpass(
    "Enter your Google AI Studio API key: "
)

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

print("Gemini API key loaded.")

Enter your Google AI Studio API key:  ········


Gemini API key loaded.


In [5]:
gemini_client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

print("Gemini connected successfully.")


Gemini connected successfully.


In [6]:
response = gemini_client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Explain weather forecasting in one simple sentence."
)

print(response.text)

Weather forecasting is the use of science and technology to predict what the weather will be like in the future.


In [7]:
def get_coordinates(city: str):

    url = "https://geocoding-api.open-meteo.com/v1/search"

    params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    response = httpx.get(
        url,
        params=params,
        timeout=10
    )

    if response.status_code != 200:
        raise Exception(
            f"Geocoding API Error "
            f"{response.status_code}: "
            f"{response.text}"
        )

    data = response.json()

    if not data.get("results"):
        raise ValueError(
            f"City '{city}' was not found."
        )

    location = data["results"][0]

    return {
        "name": location["name"],
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "country": location.get("country"),
        "timezone": location.get("timezone")
    }

In [10]:
location = get_coordinates("Bhopal")

print(
    json.dumps(
        location,
        indent=2
    )
)

{
  "name": "Bhopal",
  "latitude": 23.25469,
  "longitude": 77.40289,
  "country": "India",
  "timezone": "Asia/Kolkata"
}


In [11]:
def get_current_weather(city: str):

    location = get_coordinates(city)

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "current": (
            "temperature_2m,"
            "relative_humidity_2m,"
            "apparent_temperature,"
            "precipitation,"
            "weather_code,"
            "wind_speed_10m"
        ),
        "timezone": "auto"
    }

    response = httpx.get(
        url,
        params=params,
        timeout=10
    )

    if response.status_code != 200:
        raise Exception(
            f"Weather API Error "
            f"{response.status_code}: "
            f"{response.text}"
        )

    data = response.json()

    current = data["current"]

    return {
        "city": location["name"],
        "country": location["country"],
        "temperature_c": current["temperature_2m"],
        "feels_like_c": current["apparent_temperature"],
        "humidity_percent": current[
            "relative_humidity_2m"
        ],
        "precipitation_mm": current[
            "precipitation"
        ],
        "weather_code": current[
            "weather_code"
        ],
        "wind_speed_kmh": current[
            "wind_speed_10m"
        ],
        "time": current["time"],
        "timezone": data["timezone"]
    }

In [12]:
weather = get_current_weather(
    "Bhopal"
)

print(
    json.dumps(
        weather,
        indent=2
    )
)

{
  "city": "Bhopal",
  "country": "India",
  "temperature_c": 25.9,
  "feels_like_c": 31.4,
  "humidity_percent": 93,
  "precipitation_mm": 0.3,
  "weather_code": 55,
  "wind_speed_kmh": 7.2,
  "time": "2026-08-11T17:00",
  "timezone": "Asia/Kolkata"
}


In [13]:
def weather_code_to_text(code: int):

    codes = {

        0: "Clear sky",

        1: "Mainly clear",
        2: "Partly cloudy",
        3: "Overcast",

        45: "Fog",
        48: "Depositing rime fog",

        51: "Light drizzle",
        53: "Moderate drizzle",
        55: "Dense drizzle",

        56: "Light freezing drizzle",
        57: "Dense freezing drizzle",

        61: "Slight rain",
        63: "Moderate rain",
        65: "Heavy rain",

        66: "Light freezing rain",
        67: "Heavy freezing rain",

        71: "Slight snow",
        73: "Moderate snow",
        75: "Heavy snow",

        77: "Snow grains",

        80: "Slight rain showers",
        81: "Moderate rain showers",
        82: "Violent rain showers",

        85: "Slight snow showers",
        86: "Heavy snow showers",

        95: "Thunderstorm",

        96: "Thunderstorm with slight hail",
        99: "Thunderstorm with heavy hail"
    }

    return codes.get(
        code,
        "Unknown weather condition"
    )

In [14]:
def get_current_weather(city: str):

    location = get_coordinates(city)

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": location["latitude"],
        "longitude": location["longitude"],

        "current": (
            "temperature_2m,"
            "relative_humidity_2m,"
            "apparent_temperature,"
            "precipitation,"
            "weather_code,"
            "wind_speed_10m"
        ),

        "timezone": "auto"
    }

    response = httpx.get(
        url,
        params=params,
        timeout=10
    )

    if response.status_code != 200:
        raise Exception(
            f"Weather API Error "
            f"{response.status_code}: "
            f"{response.text}"
        )

    data = response.json()

    current = data["current"]

    return {
        "city": location["name"],
        "country": location["country"],

        "condition": weather_code_to_text(
            current["weather_code"]
        ),

        "temperature_c": current[
            "temperature_2m"
        ],

        "feels_like_c": current[
            "apparent_temperature"
        ],

        "humidity_percent": current[
            "relative_humidity_2m"
        ],

        "precipitation_mm": current[
            "precipitation"
        ],

        "wind_speed_kmh": current[
            "wind_speed_10m"
        ],

        "time": current["time"],

        "timezone": data["timezone"]
    }

In [15]:
weather = get_current_weather(
    "Bhopal"
)

print(
    json.dumps(
        weather,
        indent=2
    )
)


{
  "city": "Bhopal",
  "country": "India",
  "condition": "Dense drizzle",
  "temperature_c": 25.9,
  "feels_like_c": 31.4,
  "humidity_percent": 93,
  "precipitation_mm": 0.3,
  "wind_speed_kmh": 7.2,
  "time": "2026-08-11T17:00",
  "timezone": "Asia/Kolkata"
}


In [16]:
def get_weather_forecast(
    city: str,
    days: int = 3
):

    if days < 1:
        days = 1

    if days > 7:
        days = 7

    location = get_coordinates(city)

    url = "https://api.open-meteo.com/v1/forecast"

    params = {

        "latitude": location["latitude"],

        "longitude": location["longitude"],

        "daily": (
            "weather_code,"
            "temperature_2m_max,"
            "temperature_2m_min,"
            "precipitation_probability_max,"
            "precipitation_sum,"
            "wind_speed_10m_max"
        ),

        "forecast_days": days,

        "timezone": "auto"
    }

    response = httpx.get(
        url,
        params=params,
        timeout=10
    )

    if response.status_code != 200:
        raise Exception(
            f"Forecast API Error "
            f"{response.status_code}: "
            f"{response.text}"
        )

    data = response.json()

    daily = data["daily"]

    forecast = []

    for i in range(len(daily["time"])):

        forecast.append({

            "date": daily["time"][i],

            "condition": weather_code_to_text(
                daily["weather_code"][i]
            ),

            "max_temperature_c":
                daily["temperature_2m_max"][i],

            "min_temperature_c":
                daily["temperature_2m_min"][i],

            "rain_probability_percent":
                daily[
                    "precipitation_probability_max"
                ][i],

            "precipitation_mm":
                daily["precipitation_sum"][i],

            "max_wind_speed_kmh":
                daily["wind_speed_10m_max"][i]
        })

    return {
        "city": location["name"],
        "country": location["country"],
        "timezone": data["timezone"],
        "forecast": forecast
    }

In [17]:
forecast = get_weather_forecast(
    "Bhopal",
    3
)

print(
    json.dumps(
        forecast,
        indent=2
    )
)

{
  "city": "Bhopal",
  "country": "India",
  "timezone": "Asia/Kolkata",
  "forecast": [
    {
      "date": "2026-08-11",
      "condition": "Thunderstorm with slight hail",
      "max_temperature_c": 28.4,
      "min_temperature_c": 22.7,
      "rain_probability_percent": 100,
      "precipitation_mm": 37.3,
      "max_wind_speed_kmh": 15.5
    },
    {
      "date": "2026-08-12",
      "condition": "Thunderstorm",
      "max_temperature_c": 28.1,
      "min_temperature_c": 22.9,
      "rain_probability_percent": 95,
      "precipitation_mm": 20.2,
      "max_wind_speed_kmh": 13.3
    },
    {
      "date": "2026-08-13",
      "condition": "Thunderstorm",
      "max_temperature_c": 28.6,
      "min_temperature_c": 23.2,
      "rain_probability_percent": 94,
      "precipitation_mm": 44.5,
      "max_wind_speed_kmh": 21.5
    }
  ]
}


In [18]:
def compare_weather(
    city1: str,
    city2: str
):

    weather1 = get_current_weather(
        city1
    )

    weather2 = get_current_weather(
        city2
    )

    return {
        "city1": weather1,
        "city2": weather2
    }

In [19]:
comparison = compare_weather(
    "Bhopal",
    "Delhi"
)

print(
    json.dumps(
        comparison,
        indent=2
    )
)

{
  "city1": {
    "city": "Bhopal",
    "country": "India",
    "condition": "Dense drizzle",
    "temperature_c": 25.9,
    "feels_like_c": 31.4,
    "humidity_percent": 93,
    "precipitation_mm": 0.3,
    "wind_speed_kmh": 7.2,
    "time": "2026-08-11T17:00",
    "timezone": "Asia/Kolkata"
  },
  "city2": {
    "city": "Delhi",
    "country": "India",
    "condition": "Overcast",
    "temperature_c": 31.0,
    "feels_like_c": 38.1,
    "humidity_percent": 77,
    "precipitation_mm": 0.0,
    "wind_speed_kmh": 3.7,
    "time": "2026-08-11T17:00",
    "timezone": "Asia/Kolkata"
  }
}


In [20]:
%%writefile weather_mcp_server.py

import httpx

from mcp.server import MCPServer


mcp = MCPServer(
    "Weather MCP Server"
)


def get_coordinates(city: str):

    url = (
        "https://geocoding-api.open-meteo.com/v1/search"
    )

    params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    response = httpx.get(
        url,
        params=params,
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    if not data.get("results"):
        raise ValueError(
            f"City '{city}' not found."
        )

    location = data["results"][0]

    return {
        "name": location["name"],
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "country": location.get("country"),
        "timezone": location.get("timezone")
    }


def weather_code_to_text(code: int):

    codes = {

        0: "Clear sky",

        1: "Mainly clear",
        2: "Partly cloudy",
        3: "Overcast",

        45: "Fog",
        48: "Depositing rime fog",

        51: "Light drizzle",
        53: "Moderate drizzle",
        55: "Dense drizzle",

        61: "Slight rain",
        63: "Moderate rain",
        65: "Heavy rain",

        71: "Slight snow",
        73: "Moderate snow",
        75: "Heavy snow",

        80: "Slight rain showers",
        81: "Moderate rain showers",
        82: "Violent rain showers",

        95: "Thunderstorm",

        96: "Thunderstorm with slight hail",
        99: "Thunderstorm with heavy hail"
    }

    return codes.get(
        code,
        "Unknown"
    )


@mcp.tool()
def get_current_weather(
    city: str
) -> dict:
    """
    Get the current weather for a city.
    """

    location = get_coordinates(city)

    url = (
        "https://api.open-meteo.com/v1/forecast"
    )

    params = {

        "latitude": location["latitude"],

        "longitude": location["longitude"],

        "current": (
            "temperature_2m,"
            "relative_humidity_2m,"
            "apparent_temperature,"
            "precipitation,"
            "weather_code,"
            "wind_speed_10m"
        ),

        "timezone": "auto"
    }

    response = httpx.get(
        url,
        params=params,
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    current = data["current"]

    return {

        "city": location["name"],

        "country": location["country"],

        "condition": weather_code_to_text(
            current["weather_code"]
        ),

        "temperature_c":
            current["temperature_2m"],

        "feels_like_c":
            current["apparent_temperature"],

        "humidity_percent":
            current["relative_humidity_2m"],

        "precipitation_mm":
            current["precipitation"],

        "wind_speed_kmh":
            current["wind_speed_10m"],

        "time":
            current["time"],

        "timezone":
            data["timezone"]
    }


@mcp.tool()
def get_weather_forecast(
    city: str,
    days: int = 3
) -> dict:
    """
    Get the weather forecast for a city.
    """

    if days < 1:
        days = 1

    if days > 7:
        days = 7

    location = get_coordinates(city)

    url = (
        "https://api.open-meteo.com/v1/forecast"
    )

    params = {

        "latitude": location["latitude"],

        "longitude": location["longitude"],

        "daily": (
            "weather_code,"
            "temperature_2m_max,"
            "temperature_2m_min,"
            "precipitation_probability_max,"
            "precipitation_sum,"
            "wind_speed_10m_max"
        ),

        "forecast_days": days,

        "timezone": "auto"
    }

    response = httpx.get(
        url,
        params=params,
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    daily = data["daily"]

    forecast = []

    for i in range(len(daily["time"])):

        forecast.append({

            "date": daily["time"][i],

            "condition": weather_code_to_text(
                daily["weather_code"][i]
            ),

            "max_temperature_c":
                daily["temperature_2m_max"][i],

            "min_temperature_c":
                daily["temperature_2m_min"][i],

            "rain_probability_percent":
                daily[
                    "precipitation_probability_max"
                ][i],

            "precipitation_mm":
                daily["precipitation_sum"][i],

            "max_wind_speed_kmh":
                daily["wind_speed_10m_max"][i]
        })

    return {

        "city": location["name"],

        "country": location["country"],

        "timezone": data["timezone"],

        "forecast": forecast
    }


@mcp.tool()
def compare_weather(
    city1: str,
    city2: str
) -> dict:
    """
    Compare current weather between two cities.
    """

    weather1 = get_current_weather(city1)

    weather2 = get_current_weather(city2)

    return {
        "city1": weather1,
        "city2": weather2
    }


if __name__ == "__main__":
    mcp.run()

Overwriting weather_mcp_server.py


In [21]:
from pathlib import Path

print(
    Path(
        "weather_mcp_server.py"
    ).read_text()
)


import httpx

from mcp.server import MCPServer


mcp = MCPServer(
    "Weather MCP Server"
)


def get_coordinates(city: str):

    url = (
        "https://geocoding-api.open-meteo.com/v1/search"
    )

    params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    response = httpx.get(
        url,
        params=params,
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    if not data.get("results"):
        raise ValueError(
            f"City '{city}' not found."
        )

    location = data["results"][0]

    return {
        "name": location["name"],
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "country": location.get("country"),
        "timezone": location.get("timezone")
    }


def weather_code_to_text(code: int):

    codes = {

        0: "Clear sky",

        1: "Mainly clear",
        2: "Partly cloudy",
        3: "Overcast",



In [22]:
weather_tools = types.Tool(

    function_declarations=[

        types.FunctionDeclaration(

            name="get_current_weather",

            description=(
                "Get the current weather "
                "for a city."
            ),

            parameters={

                "type": "object",

                "properties": {

                    "city": {
                        "type": "string",
                        "description":
                            "Name of the city."
                    }
                },

                "required": [
                    "city"
                ]
            }
        ),

        types.FunctionDeclaration(

            name="get_weather_forecast",

            description=(
                "Get the weather forecast "
                "for a city for 1 to 7 days."
            ),

            parameters={

                "type": "object",

                "properties": {

                    "city": {
                        "type": "string"
                    },

                    "days": {
                        "type": "integer",
                        "description":
                            "Number of forecast days."
                    }
                },

                "required": [
                    "city"
                ]
            }
        ),

        types.FunctionDeclaration(

            name="compare_weather",

            description=(
                "Compare the current weather "
                "between two cities."
            ),

            parameters={

                "type": "object",

                "properties": {

                    "city1": {
                        "type": "string"
                    },

                    "city2": {
                        "type": "string"
                    }
                },

                "required": [
                    "city1",
                    "city2"
                ]
            }
        )
    ]
)

print("Weather tools registered with Gemini.")

Weather tools registered with Gemini.


In [23]:
def execute_weather_tool(
    function_name,
    arguments
):

    if function_name == "get_current_weather":

        return get_current_weather(
            arguments["city"]
        )

    elif function_name == "get_weather_forecast":

        days = arguments.get(
            "days",
            3
        )

        return get_weather_forecast(
            arguments["city"],
            days
        )

    elif function_name == "compare_weather":

        return compare_weather(
            arguments["city1"],
            arguments["city2"]
        )

    else:

        raise ValueError(
            f"Unknown tool: {function_name}"
        )

In [25]:
user_message = """
What is the current weather in Bhopal?
"""

response = gemini_client.models.generate_content(

    model="gemini-3.5-flash",

    contents=user_message,

    config=types.GenerateContentConfig(
        tools=[weather_tools]
    )
)

for part in response.candidates[0].content.parts:

    if part.function_call:

        print(
            "Tool:",
            part.function_call.name
        )

        print(
            "Arguments:",
            dict(part.function_call.args)
        )

Tool: get_current_weather
Arguments: {'city': 'Bhopal'}


In [26]:
function_call = None

for part in response.candidates[0].content.parts:

    if part.function_call:

        function_call = part.function_call

        break


if function_call:

    tool_name = function_call.name

    arguments = dict(
        function_call.args
    )

    tool_result = execute_weather_tool(
        tool_name,
        arguments
    )

    print(
        json.dumps(
            tool_result,
            indent=2
        )
    )

else:

    print(
        response.text
    )

{
  "city": "Bhopal",
  "country": "India",
  "condition": "Dense drizzle",
  "temperature_c": 25.9,
  "feels_like_c": 31.4,
  "humidity_percent": 93,
  "precipitation_mm": 0.3,
  "wind_speed_kmh": 7.2,
  "time": "2026-08-11T17:00",
  "timezone": "Asia/Kolkata"
}


In [27]:
tool_response = types.Part.from_function_response(

    name=function_call.name,

    response={
        "result": tool_result
    }
)

print("Weather data sent back to Gemini.")

Weather data sent back to Gemini.


In [35]:
final_response = gemini_client.models.generate_content(

    model="gemini-3.5-flash",

    contents=[

        user_message,

        response.candidates[0].content,

        types.Content(

            role="user",

            parts=[
                tool_response
            ]
        )
    ],

    config=types.GenerateContentConfig(
        tools=[weather_tools]
    )
)

print(final_response.text)

The current weather in Bhopal, India is 25.9°C (feels like 31.4°C) with dense drizzle. The humidity is at 93% and the wind is blowing at 7.2 km/h.


In [36]:
async def ask_weather_agent(
    user_message: str
):

    response = gemini_client.models.generate_content(

        model="gemini-3.5-flash",

        contents=user_message,

        config=types.GenerateContentConfig(
            tools=[weather_tools]
        )
    )

    function_call = None

    for part in response.candidates[0].content.parts:

        if part.function_call:

            function_call = part.function_call

            break

    if not function_call:

        return response.text

    tool_name = function_call.name

    arguments = dict(
        function_call.args
    )

    tool_result = execute_weather_tool(
        tool_name,
        arguments
    )

    tool_response = types.Part.from_function_response(

        name=tool_name,

        response={
            "result": tool_result
        }
    )

    final_response = gemini_client.models.generate_content(

        model="gemini-3.5-flash",

        contents=[

            user_message,

            response.candidates[0].content,

            types.Content(

                role="user",

                parts=[
                    tool_response
                ]
            )
        ],

        config=types.GenerateContentConfig(
            tools=[weather_tools]
        )
    )

    return final_response.text

In [37]:
answer = await ask_weather_agent(
    "What's the weather in Bhopal right now?"
)

print(answer)

The current weather in Bhopal is a dense drizzle with a temperature of 25.9°C (feels like 31.4°C). The humidity is high at 93%, and there is a light wind of 7.2 km/h.


In [38]:
answer = await ask_weather_agent(
    "What will the weather be like "
    "in Bhopal for the next 3 days?"
)

print(answer)

The weather forecast for Bhopal over the next 3 days is as follows:

*   **August 11, 2026**: Expect a **thunderstorm with slight hail**. The temperature will range from a low of 22.7°C to a high of 28.4°C. There is a 100% chance of rain with about 37.3 mm of precipitation. Winds will be around 15.5 km/h.
*   **August 12, 2026**: A **thunderstorm** is expected with temperatures between 22.9°C and 28.1°C. There is a 95% chance of rain (20.2 mm) and winds of 13.3 km/h.
*   **August 13, 2026**: Another **thunderstorm** is forecasted with a low of 23.2°C and a high of 28.6°C. Rain probability is high at 94% with significant precipitation of 44.5 mm. Winds will pick up to about 21.5 km/h.


In [40]:
answer = await ask_weather_agent(
    "Is there a chance of rain in Bhopal "
    "over the next 3 days?"
)

print(answer)

Yes, there is a very high chance of rain in Bhopal over the next 3 days:

*   **Day 1 (Aug 11):** 100% chance of rain with thunderstorms and slight hail (37.3 mm of precipitation).
*   **Day 2 (Aug 12):** 95% chance of rain with thunderstorms (20.2 mm of precipitation).
*   **Day 3 (Aug 13):** 94% chance of rain with thunderstorms (44.5 mm of precipitation).


In [41]:
answer = await ask_weather_agent(
    "Compare the current weather "
    "in Bhopal and Delhi."
)

print(answer)

The current weather in Bhopal and Delhi is as follows:

*   **Bhopal:** It is currently experiencing a **dense drizzle** with a temperature of **25.7°C** (feels like 31.2°C). The humidity is very high at **94%**, and there is a light wind of 7 km/h.
*   **Delhi:** The weather is **partly cloudy** and warmer, with a temperature of **31.0°C** (feels like 38.1°C). The humidity is **77%**, with a gentle breeze of 3.1 km/h.


In [42]:
answer = await ask_weather_agent(
    """
    I'm planning a morning walk in Bhopal.
    Check the current weather and tell me
    whether it looks comfortable for walking.
    Give me practical advice.
    """
)

print(answer)

Currently, the weather in Bhopal is not ideal for a comfortable walk. Here are the details and some practical advice:

### **Current Weather in Bhopal**
* **Condition:** Dense drizzle
* **Temperature:** 25.7°C (but it feels like a sticky **31.2°C**)
* **Humidity:** Extremely high at **94%**
* **Wind:** Light breeze at 7 km/h

### **Is it comfortable for walking?**
**Not particularly.** While the actual temperature is mild, the **94% humidity** combined with the "feels like" temperature of over 31°C will make it feel very muggy, warm, and sweaty. Additionally, the **dense drizzle** means you are likely to get wet.

### **Practical Advice**
* **Gear Up:** If you still decide to go, definitely carry an **umbrella** or wear a light, waterproof **rain jacket**. 
* **Wear the Right Clothes:** Opt for lightweight, moisture-wicking, and quick-drying athletic wear. Avoid cotton, as it will absorb the humidity and drizzle, becoming heavy and uncomfortable.
* **Stay Hydrated:** High humidity prev

In [43]:
answer = await ask_weather_agent(
    """
    I'm travelling to Mumbai tomorrow.
    Check the forecast and tell me:
    1. Whether I should carry an umbrella
    2. What kind of clothes would be comfortable
    3. Whether I should expect strong wind
    """
)

print(answer)

Based on the weather forecast for Mumbai over the next two days, here are the answers to your questions:

1. **Should you carry an umbrella?**
   **Yes, absolutely.** The forecast shows a **100% chance of rain** for both days, with conditions ranging from thunderstorms to slight rain showers. You will definitely need an umbrella or a raincoat.

2. **What kind of clothes would be comfortable?**
   The temperatures will be warm and humid, ranging from a low of **25°C (77°F)** to a high of **29°C (84°F)**. 
   * **Recommendation:** Wear **light, breathable, and quick-drying cotton or synthetic clothes**. Since it will be rainy and humid, heavy fabrics like denim might take a long time to dry and feel uncomfortable. Waterproof footwear or sandals are also highly recommended.

3. **Should you expect strong wind?**
   **No, not particularly strong wind.** The maximum wind speeds are expected to be around **17 to 19 km/h** (approx. 10-12 mph). This is classified as a gentle to moderate breeze

In [44]:
answer = await ask_weather_agent(
    """
    I want to play cricket outdoors tomorrow
    in Bhopal.

    Check the weather forecast and tell me
    whether the conditions look suitable.
    Mention rain probability, temperature,
    and wind.
    """
)

print(answer)

Based on the weather forecast for tomorrow (August 12, 2026) in Bhopal, the conditions **do not look suitable** for playing cricket outdoors. 

Here are the expected weather details:
*   **Condition:** Thunderstorms
*   **Rain Probability:** 95% (with approximately 20.2 mm of rain expected)
*   **Temperature:** A high of 28.1°C and a low of 22.9°C
*   **Wind:** Max wind speeds around 13.3 km/h

With a very high chance of thunderstorms and significant rain, it is highly recommended to postpone your cricket match or look for an indoor alternative.


In [45]:
!pip install streamlit


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [47]:
%%writefile weather_tools.py

import httpx


def get_coordinates(city: str):

    url = "https://geocoding-api.open-meteo.com/v1/search"

    params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    response = httpx.get(
        url,
        params=params,
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    if not data.get("results"):
        raise ValueError(
            f"City '{city}' not found."
        )

    location = data["results"][0]

    return {
        "name": location["name"],
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "country": location.get("country"),
        "timezone": location.get("timezone")
    }


def weather_code_to_text(code: int):

    codes = {

        0: "Clear sky",

        1: "Mainly clear",
        2: "Partly cloudy",
        3: "Overcast",

        45: "Fog",
        48: "Depositing rime fog",

        51: "Light drizzle",
        53: "Moderate drizzle",
        55: "Dense drizzle",

        56: "Light freezing drizzle",
        57: "Dense freezing drizzle",

        61: "Slight rain",
        63: "Moderate rain",
        65: "Heavy rain",

        66: "Light freezing rain",
        67: "Heavy freezing rain",

        71: "Slight snow",
        73: "Moderate snow",
        75: "Heavy snow",

        77: "Snow grains",

        80: "Slight rain showers",
        81: "Moderate rain showers",
        82: "Violent rain showers",

        85: "Slight snow showers",
        86: "Heavy snow showers",

        95: "Thunderstorm",

        96: "Thunderstorm with slight hail",
        99: "Thunderstorm with heavy hail"
    }

    return codes.get(
        code,
        "Unknown weather condition"
    )


def get_current_weather(city: str):

    location = get_coordinates(city)

    url = "https://api.open-meteo.com/v1/forecast"

    params = {

        "latitude": location["latitude"],

        "longitude": location["longitude"],

        "current": (
            "temperature_2m,"
            "relative_humidity_2m,"
            "apparent_temperature,"
            "precipitation,"
            "weather_code,"
            "wind_speed_10m"
        ),

        "timezone": "auto"
    }

    response = httpx.get(
        url,
        params=params,
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    current = data["current"]

    return {

        "city": location["name"],

        "country": location["country"],

        "condition": weather_code_to_text(
            current["weather_code"]
        ),

        "temperature_c":
            current["temperature_2m"],

        "feels_like_c":
            current["apparent_temperature"],

        "humidity_percent":
            current["relative_humidity_2m"],

        "precipitation_mm":
            current["precipitation"],

        "wind_speed_kmh":
            current["wind_speed_10m"],

        "time":
            current["time"],

        "timezone":
            data["timezone"]
    }


def get_weather_forecast(
    city: str,
    days: int = 3
):

    if days < 1:
        days = 1

    if days > 7:
        days = 7

    location = get_coordinates(city)

    url = "https://api.open-meteo.com/v1/forecast"

    params = {

        "latitude": location["latitude"],

        "longitude": location["longitude"],

        "daily": (
            "weather_code,"
            "temperature_2m_max,"
            "temperature_2m_min,"
            "precipitation_probability_max,"
            "precipitation_sum,"
            "wind_speed_10m_max"
        ),

        "forecast_days": days,

        "timezone": "auto"
    }

    response = httpx.get(
        url,
        params=params,
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    daily = data["daily"]

    forecast = []

    for i in range(len(daily["time"])):

        forecast.append({

            "date": daily["time"][i],

            "condition": weather_code_to_text(
                daily["weather_code"][i]
            ),

            "max_temperature_c":
                daily["temperature_2m_max"][i],

            "min_temperature_c":
                daily["temperature_2m_min"][i],

            "rain_probability_percent":
                daily[
                    "precipitation_probability_max"
                ][i],

            "precipitation_mm":
                daily["precipitation_sum"][i],

            "max_wind_speed_kmh":
                daily["wind_speed_10m_max"][i]
        })

    return {

        "city": location["name"],

        "country": location["country"],

        "timezone": data["timezone"],

        "forecast": forecast
    }


def compare_weather(
    city1: str,
    city2: str
):

    weather1 = get_current_weather(city1)

    weather2 = get_current_weather(city2)

    return {
        "city1": weather1,
        "city2": weather2
    }

Writing weather_tools.py


In [48]:
from pathlib import Path

print(Path("weather_tools.py").exists())
print(Path("weather_tools.py").absolute())

True
/Users/aashishmewada/Desktop/MCP/weather_tools.py


In [49]:
from weather_tools import (
    get_current_weather,
    get_weather_forecast,
    compare_weather
)

print("Weather tools imported successfully!")

Weather tools imported successfully!


In [50]:
weather = get_current_weather("Bhopal")

print(weather)

{'city': 'Bhopal', 'country': 'India', 'condition': 'Dense drizzle', 'temperature_c': 25.7, 'feels_like_c': 31.2, 'humidity_percent': 94, 'precipitation_mm': 0.3, 'wind_speed_kmh': 7.0, 'time': '2026-08-11T17:15', 'timezone': 'Asia/Kolkata'}


In [51]:
%%writefile app.py

import streamlit as st

from weather_tools import (
    get_current_weather,
    get_weather_forecast,
    compare_weather
)


st.set_page_config(
    page_title="Weather AI",
    page_icon="🌦️",
    layout="wide"
)


st.title("🌦️ Weather AI Assistant")

st.write(
    "Get current weather and forecasts."
)


city = st.text_input(
    "Enter city",
    "Bhopal"
)


if st.button("Get Weather"):

    try:

        weather = get_current_weather(city)

        col1, col2, col3, col4 = st.columns(4)

        with col1:
            st.metric(
                "Temperature",
                f"{weather['temperature_c']} °C"
            )

        with col2:
            st.metric(
                "Feels Like",
                f"{weather['feels_like_c']} °C"
            )

        with col3:
            st.metric(
                "Humidity",
                f"{weather['humidity_percent']}%"
            )

        with col4:
            st.metric(
                "Wind",
                f"{weather['wind_speed_kmh']} km/h"
            )

        st.subheader(
            weather["condition"]
        )

        st.write(
            f"📍 {weather['city']}, "
            f"{weather['country']}"
        )

    except Exception as e:

        st.error(
            f"Error: {e}"
        )

Writing app.py


In [52]:
from pathlib import Path

print(list(Path(".").glob("*")))

[PosixPath('weather_tools.py'), PosixPath('Untitled1.ipynb'), PosixPath('Untitled.ipynb'), PosixPath('weather_mcp_server.py'), PosixPath('__pycache__'), PosixPath('weather.ipynb'), PosixPath('currency_mcp_server.py'), PosixPath('app.py'), PosixPath('.ipynb_checkpoints'), PosixPath('.git')]


In [53]:
print(Path("app.py").read_text())


import streamlit as st

from weather_tools import (
    get_current_weather,
    get_weather_forecast,
    compare_weather
)


st.set_page_config(
    page_title="Weather AI",
    page_icon="🌦️",
    layout="wide"
)


st.title("🌦️ Weather AI Assistant")

st.write(
    "Get current weather and forecasts."
)


city = st.text_input(
    "Enter city",
    "Bhopal"
)


if st.button("Get Weather"):

    try:

        weather = get_current_weather(city)

        col1, col2, col3, col4 = st.columns(4)

        with col1:
            st.metric(
                "Temperature",
                f"{weather['temperature_c']} °C"
            )

        with col2:
            st.metric(
                "Feels Like",
                f"{weather['feels_like_c']} °C"
            )

        with col3:
            st.metric(
                "Humidity",
                f"{weather['humidity_percent']}%"
            )

        with col4:
            st.metric(
                "Wind",
                f"{weather

In [54]:
import streamlit as st